In [91]:
# Importing the necessary libraries
import pandas as pd
import numpy as np

In [3]:
# Data loading with potential error handling
try:
    df = pd.read_csv('data.csv')
    print('The data has been uploaded')
except Exception as e:
    print('Download error occurred', e)
    raise

The data has been uploaded


In [92]:
df.head()

,children,days_employed,dob_years,education,family_status,gender,income_type,debt,total_income,purpose
0,1,8437.673028,42.0,высшее,женат / замужем,f,сотрудник,0,253875.639453,покупка жилья
1,1,4024.803754,36.0,среднее,женат / замужем,f,сотрудник,0,112080.014102,приобретение автомобиля
2,0,5623.422610,33.0,среднее,женат / замужем,m,сотрудник,0,145885.952297,покупка жилья
3,3,4124.747207,32.0,среднее,женат / замужем,m,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53.0,среднее,гражданский брак,f,пенсионер,0,158616.077870,сыграть свадьбу


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  str    
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  str    
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  str    
 8   income_type       21525 non-null  str    
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  str    
dtypes: float64(2), int64(5), str(5)
memory usage: 4.1 MB


In [9]:
display(df.describe(include = 'all'))

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
count,21525.000000,19351.000000,21525.000000,21525,21525.000000,21525,21525.000000,21525,21525,21525.000000,1.935100e+04,21525
unique,NaN,NaN,NaN,15,NaN,5,NaN,3,8,NaN,NaN,38
top,NaN,NaN,NaN,среднее,NaN,женат / замужем,NaN,F,сотрудник,NaN,NaN,свадьба
freq,NaN,NaN,NaN,13750,NaN,12380,NaN,14236,11119,NaN,NaN,797
mean,0.538908,63046.497661,43.293380,NaN,0.817236,NaN,0.972544,NaN,NaN,0.080883,1.674223e+05,NaN
std,1.381587,140827.311974,12.574584,NaN,0.548138,NaN,1.420324,NaN,NaN,0.272661,1.029716e+05,NaN
min,-1.000000,-18388.949901,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,0.000000,2.066726e+04,NaN
25%,0.000000,-2747.423625,33.000000,NaN,1.000000,NaN,0.000000,NaN,NaN,0.000000,1.030532e+05,NaN
50%,0.000000,-1203.369529,42.000000,NaN,1.000000,NaN,0.000000,NaN,NaN,0.000000,1.450179e+05,NaN
75%,1.000000,-291.095954,53.000000,NaN,1.000000,NaN,1.000000,NaN,NaN,0.000000,2.034351e+05,NaN


In [10]:
missing_data = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_ratio': df.isna().mean()
}).sort_values('missing_ratio', ascending = False)

The data contains gaps in at least two columns "days_employed" and "total_income". The "days_employed" column
displays the length of employment in days. Gaps have been appeared due to the lack of information employment length, 
errors during data export, or merging of sources. The "total_income" column displays the monthly income. Gaps have been appeared due to the client didn't specify the income or the data were not transferred.

In [93]:
display(missing_data)

,missing_count,missing_ratio
days_employed,2174,0.100999
total_income,2174,0.100999
children,0,0.000000
dob_years,0,0.000000
education,0,0.000000
education_id,0,0.000000
family_status,0,0.000000
family_status_id,0,0.000000
gender,0,0.000000
income_type,0,0.000000


For quantitative variables, filling with the median is a good solution because the median is resistant to
outliers and anomalous values, unlike the mean. It doesn't significantly distort the distribution and preserves 
the observations.

In [11]:
display(missing_data[missing_data['missing_count'] > 0])

,missing_count,missing_ratio
days_employed,2174,0.100999
total_income,2174,0.100999


In [12]:
df['total_income'] = df['total_income'].fillna(df['total_income'].median())

In [13]:
df['days_employed'] = df['days_employed'].abs()
df['days_employed'] = df['days_employed'].fillna(df['days_employed'].median())

In [14]:
df.loc[df['dob_years'] <= 0, 'dob_years'] = np.nan
df['dob_years'] = df['dob_years'].fillna(df['dob_years'].median())
df.loc[df['children'] < 0, 'children'] = 0

In [15]:
print(df[['days_employed', 'total_income']].isna().sum())

days_employed    0
total_income     0
dtype: int64


In [16]:
text_cols = ['education', 'family_status', 'gender', 'income_type', 'purpose']

In [17]:
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [18]:
before_drop_dubl = df.shape[0]
df = df.drop_duplicates().reset_index(drop = True)
after_drop_dubl = df.shape[0]

In [19]:
df = df.drop(columns = ['education_id', 'family_status_id'], errors = 'ignore')

In [23]:
education = (
    df[['education']]
        .drop_duplicates()
        .sort_values('education')
        .reset_index(drop = True)
)

In [24]:
education['education_id'] = education.index

In [25]:
display(education)

,education,education_id
0,высшее,0
1,начальное,1
2,неоконченное высшее,2
3,среднее,3
4,ученая степень,4


In [59]:
# Словарь для семейного положения
family_status = (
    df[['family_status']]
    .drop_duplicates()
    .sort_values('family_status')
    .reset_index(drop = True)
)

In [60]:
family_status['family_status_id'] = family_status.index

In [61]:
display(family_status)

,family_status,family_status_id
0,в разводе,0
1,вдовец / вдова,1
2,гражданский брак,2
3,женат / замужем,3
4,не женат / не замужем,4


In [62]:
df1 = df.merge(education, on = 'education', how = 'left')
df1 = df1.merge(family_status, on = 'family_status', how = 'left')
df1 = df1.drop(columns = ['education', 'family_status'])

In [63]:
display(df1.head())

,children,days_employed,dob_years,gender,income_type,debt,total_income,purpose,education_id,family_status_id
0,1,8437.673028,42.0,f,сотрудник,0,253875.639453,покупка жилья,0,3
1,1,4024.803754,36.0,f,сотрудник,0,112080.014102,приобретение автомобиля,3,3
2,0,5623.422610,33.0,m,сотрудник,0,145885.952297,покупка жилья,3,3
3,3,4124.747207,32.0,m,сотрудник,0,267628.550329,дополнительное образование,3,3
4,0,340266.072047,53.0,f,пенсионер,0,158616.077870,сыграть свадьбу,3,2


In [64]:
df1['total_income_category'] = pd.cut(
    df['total_income'],
    bins = [0, 30000, 50000, 200000, 1000000, np.inf],
    labels = ['E', 'D', 'C', 'B', 'A'],
    right = True,
    include_lowest = True
)

In [65]:
display(df1['total_income_category'].value_counts().sort_index())

total_income_category
E       22
D      350
C    16016
B     5041
A       25
Name: count, dtype: int64

In [66]:
def get_purpose_category(purpose):
    try:
        p = str(purpose).lower().strip()
        if 'автомобил' in p:
            return 'операции с автомобилем'
        elif 'свадьб' in p:
            return 'проведение свадьбы'
        elif ('образован' in p) or ('образовани' in p):
            return 'получение образования'
        elif ('недвижым' in p) or ('жил' in p) or ('ремонт' in p):
            return 'операции с недвижимостью'
        else:
            return 'прочее'
    except Exception as e:
        return 'прочее'

In [73]:
df1['purpose_category'] = df['purpose'].apply(get_purpose_category)

In [82]:
df1['purpose_category'].value_counts()

purpose_category
операции с недвижимостью    5690
прочее                      5121
операции с автомобилем      4306
получение образования       4013
проведение свадьбы          2324
Name: count, dtype: int64

In [78]:
print('Общая доля просрочек:', df['debt'].mean())

Общая доля просрочек: 0.08115036822970076


In [79]:
children_pivot = pd.pivot_table(
    df1,
    values = 'debt',
    index = 'children',
    aggfunc = ['count', 'mean']
).sort_index()

In [80]:
display(children_pivot)

,count,mean
,debt,debt
children,,
0,14138,0.075258
1,4808,0.092346
2,2052,0.094542
3,330,0.081818
4,41,0.097561
5,9,0.000000
20,76,0.105263


In [81]:
family_table = pd.pivot_table(
    df1,
    values = 'debt',
    index = 'family_status_id',
    aggfunc = ['count', 'mean']
).sort_index()

In [83]:
display(family_table)

,count,mean
,debt,debt
family_status_id,,
0,1195,0.071130
1,959,0.065693
2,4151,0.093471
3,12339,0.075452
4,2810,0.097509


In [84]:
income_pivot = pd.pivot_table(
    df1,
    values = 'debt',
    index = 'total_income_category',
    aggfunc = ['count', 'mean'],
    observed = False
).sort_index()

In [87]:
display(income_pivot)

,count,mean
,debt,debt
total_income_category,,
E,22,0.090909
D,350,0.060000
C,16016,0.084915
B,5041,0.070621
A,25,0.080000


In [88]:
purpose_pivot = pd.pivot_table(
    df1,
    values = 'debt',
    index = 'purpose_category',
    aggfunc = ['count', 'mean']
).sort_values(('mean', 'debt'), ascending = False)

In [89]:
display(purpose_pivot)

,count,mean
,debt,debt
purpose_category,,
операции с автомобилем,4306,0.093590
получение образования,4013,0.092200
проведение свадьбы,2324,0.080034
прочее,5121,0.075181
операции с недвижимостью,5690,0.069772


In [90]:
print('Explicit dublicates were detected and removed:', before_drop_dubl - after_drop_dubl)

Explicit dublicates were detected and removed: 71


We look at how the share of overdue payments changes from income category E to A. If the share of overdue
payments is lower in higher categories, the income may be related to borrower's realibility. However, we need 
to take into account the size of the categories and possible outliers.

### General Results

During the study, the data were cleaned. Missing values in "total_income" and "days_employed" were hundled and negative employment duration values were replaced with absolute values as artifacts. Missing values were filled with medians.
Text categories were brought to a single case and dublicated were removed. Dictationaries for "education_id" and "family_status_id", income categories and credit purpose categories were created. 

Based on the results of the summary tables, I drew the following conclutions:
- 1.) Number of children and marital status show differences in proportion of overdue payments between groups;
- 2.) The income level is also assosiated with the proportion of overdie payments, specifically, the risk differs across different
income categories;
- 3.) The purpose of the loan affects the observed propotion of overdie payments.

The result obtained should be interpreted as statistical patterns, not as strict case-and-effect relationships. For credit scoring, these factors may be useful, but they need to be verified using additionl data and models that control for other factors.